<a href="https://colab.research.google.com/github/kasrasa/ViT-VLM-experiments/blob/VLM-Experiments/VLM_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers torch
!pip install -q timm
!pip install -q evaluate
!pip install -q peft
!pip install --upgrade -q torchao
!pip install -q scikit-learn
!pip install -q matplotlib seaborn accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 49.7 MB/s eta 0:00:00


In [2]:
import os

# Set to True to always fine-tune, False to load existing checkpoints if available
FORCE_FINE_TUNING = True

In [3]:
from transformers import DefaultDataCollator

data_collator = DefaultDataCollator()

In [4]:
import evaluate
accuracy = evaluate.load("accuracy")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [5]:
from torchvision.transforms import (
    RandomResizedCrop,
    Resize,
    CenterCrop,
    Compose,
    Normalize,
    ToTensor,
)

def get_image_size(image_processor):
    if "shortest_edge" in image_processor.size:
        return image_processor.size["shortest_edge"]
    return image_processor.size["height"]

def apply_transforms(examples, image_processor, is_train=True):
    image_size = get_image_size(image_processor)
    normalize = Normalize(
        mean=image_processor.image_mean,
        std=image_processor.image_std,
    )

    if is_train:
        transform = Compose([
            RandomResizedCrop(image_size),
            ToTensor(),
            normalize,
        ])
    else:
        transform = Compose([
            Resize(image_size),
            CenterCrop(image_size),
            ToTensor(),
            normalize,
        ])

    examples["pixel_values"] = [
        transform(img.convert("RGB")) for img in examples["image"]
    ]
    del examples["image"]
    return examples

In [6]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def softmax_np(logits):
    """
    Numerically stable softmax.
    """
    logits = logits - np.max(logits, axis=1, keepdims=True)
    exp_logits = np.exp(logits)
    return exp_logits / np.sum(exp_logits, axis=1, keepdims=True)


def expected_calibration_error(probs, labels, n_bins=10):
    """
    Expected Calibration Error.

    If the model says it is 80% confident, we want it to be correct
    about 80% of the time. ECE measures this mismatch.
    """
    confidences = np.max(probs, axis=1)
    predictions = np.argmax(probs, axis=1)
    correct = predictions == labels

    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0

    for i in range(n_bins):
        lower = bin_edges[i]
        upper = bin_edges[i + 1]

        if i == 0:
            in_bin = (confidences >= lower) & (confidences <= upper)
        else:
            in_bin = (confidences > lower) & (confidences <= upper)

        prop_in_bin = np.mean(in_bin)

        if prop_in_bin > 0:
            avg_confidence = np.mean(confidences[in_bin])
            avg_accuracy = np.mean(correct[in_bin])
            ece += prop_in_bin * abs(avg_confidence - avg_accuracy)

    return float(ece)


def compute_metrics(eval_pred):
    """
    Metrics used during Trainer evaluation.

    Includes:
    - accuracy
    - macro F1
    - weighted F1
    - top-5 accuracy
    - confidence statistics
    - ECE calibration metric
    - NLL
    - multiclass Brier score
    """
    logits = eval_pred.predictions

    # Some Hugging Face models may return predictions as a tuple.
    if isinstance(logits, tuple):
        logits = logits[0]

    labels = eval_pred.label_ids
    predictions = np.argmax(logits, axis=1)
    probs = softmax_np(logits)

    num_classes = probs.shape[1]
    k = min(5, num_classes)

    top_k_predictions = np.argsort(probs, axis=1)[:, -k:]
    top_k_accuracy = np.mean([
        labels[i] in top_k_predictions[i]
        for i in range(len(labels))
    ])

    top1_confidence = np.max(probs, axis=1)

    sorted_probs = np.sort(probs, axis=1)
    top2_confidence = sorted_probs[:, -2] if num_classes > 1 else np.zeros_like(top1_confidence)
    top1_top2_margin = top1_confidence - top2_confidence

    correct_mask = predictions == labels
    wrong_mask = ~correct_mask

    correct_mean_confidence = float(np.mean(top1_confidence[correct_mask])) if np.any(correct_mask) else 0.0
    wrong_mean_confidence = float(np.mean(top1_confidence[wrong_mask])) if np.any(wrong_mask) else 0.0

    # Negative log likelihood for the true class
    eps = 1e-12
    true_class_probs = probs[np.arange(len(labels)), labels]
    nll = -np.mean(np.log(np.clip(true_class_probs, eps, 1.0)))

    # Multiclass Brier score
    one_hot = np.zeros_like(probs)
    one_hot[np.arange(len(labels)), labels] = 1.0
    brier_score = np.mean(np.sum((probs - one_hot) ** 2, axis=1))

    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(labels, predictions, average="macro", zero_division=0),
        "weighted_f1": f1_score(labels, predictions, average="weighted", zero_division=0),

        # Keep this alias so your existing summary table does not break
        "f1": f1_score(labels, predictions, average="weighted", zero_division=0),

        "top_5_accuracy": float(top_k_accuracy),
        "mean_top1_confidence": float(np.mean(top1_confidence)),
        "mean_top1_top2_margin": float(np.mean(top1_top2_margin)),
        "correct_mean_confidence": correct_mean_confidence,
        "wrong_mean_confidence": wrong_mean_confidence,
        "ece": expected_calibration_error(probs, labels, n_bins=10),
        "nll": float(nll),
        "brier_score": float(brier_score),
    }

In [7]:
def count_parameters(model):
    """
    Counts and displays the number of trainable parameters in a PyTorch model.
    """
    num_params_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Number of trainable parameters: {num_params_trainable:,} ({num_params_trainable / 1e6:.2f} million)")
    print(f"Total parameters: {total_params:,} ({total_params / 1e6:.2f} million)")


In [8]:

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np
import pandas as pd

class ClassificationLogitsOutput:
    """
    Minimal prediction container compatible with compute_metrics().

    Both ConvNeXt and the VLM are converted to this format:
        predictions: np.ndarray of shape [num_samples, num_classes]
        label_ids:   np.ndarray of shape [num_samples]
        metrics:     optional dict, e.g. {'test_loss': ...}
    """
    def __init__(self, predictions, label_ids, metrics=None):
        self.predictions = predictions
        self.label_ids = label_ids
        self.metrics = metrics or {}


def get_trainer_logits_and_labels(trainer, dataset):
    """
    Extracts class logits and labels from a Hugging Face Trainer model.
    This is used for ConvNeXt and any other standard image classifier.
    """
    predictions_output = trainer.predict(dataset)

    logits = predictions_output.predictions
    if isinstance(logits, tuple):
        logits = logits[0]

    labels = predictions_output.label_ids
    eval_loss = predictions_output.metrics.get("test_loss", None)

    return np.asarray(logits), np.asarray(labels), eval_loss


def evaluate_logits_and_plot(
    logits,
    labels,
    id2label_mapping,
    dataset_name,
    num_labels,
    model_display_name,
    training_time=None,
    eval_loss=None,
    normalize_cm=False,
    extra_prediction_columns=None,
):
    """
    Centralized classification evaluation for any model that can produce
    class logits over the same label space.

    ConvNeXt output:
        logits from trainer.predict(...)

    VLM output:
        candidate class scores/logits computed from image-conditioned
        label log-likelihoods over the same 20 class labels.

    Because both paths end in [N, C] logits, the same metrics are used:
    accuracy, macro/weighted F1, top-5 accuracy, confidence, margin,
    ECE, NLL, Brier score, confusion matrix, and per-class report.
    """
    print(f"\n--- Evaluating {model_display_name} on {dataset_name} ---")

    logits = np.asarray(logits)
    labels = np.asarray(labels).astype(int)

    assert logits.ndim == 2, f"Expected logits with shape [N, C], got {logits.shape}"
    assert logits.shape[0] == labels.shape[0], "Number of logits and labels does not match."
    assert logits.shape[1] == num_labels, f"Expected {num_labels} classes, got {logits.shape[1]}"

    predictions_output = ClassificationLogitsOutput(
        predictions=logits,
        label_ids=labels,
        metrics={"test_loss": eval_loss} if eval_loss is not None else {},
    )

    metrics = compute_metrics(predictions_output)
    if eval_loss is not None:
        metrics["eval_loss"] = float(eval_loss)

    predicted_labels = np.argmax(logits, axis=1)
    probs = softmax_np(logits)

    print("\nCore metrics:")
    print(f"Accuracy:                {metrics['accuracy']:.4f}")
    print(f"Macro F1:                {metrics['macro_f1']:.4f}")
    print(f"Weighted F1:             {metrics['weighted_f1']:.4f}")
    print(f"Top-5 Accuracy:          {metrics['top_5_accuracy']:.4f}")

    if eval_loss is not None:
        print(f"Eval Loss:               {metrics['eval_loss']:.4f}")

    print("\nConfidence / calibration metrics:")
    print(f"Mean Top-1 Confidence:   {metrics['mean_top1_confidence']:.4f}")
    print(f"Mean Top1-Top2 Margin:   {metrics['mean_top1_top2_margin']:.4f}")
    print(f"Correct Mean Confidence: {metrics['correct_mean_confidence']:.4f}")
    print(f"Wrong Mean Confidence:   {metrics['wrong_mean_confidence']:.4f}")
    print(f"ECE:                     {metrics['ece']:.4f}")
    print(f"NLL:                     {metrics['nll']:.4f}")
    print(f"Brier Score:             {metrics['brier_score']:.4f}")

    if training_time is not None:
        print(f"\nTraining time for {model_display_name}: {training_time:.2f} seconds")

    print("\nLabel sanity check:")
    print(f"Number of unique true labels:      {len(np.unique(labels))}")
    print(f"Number of unique predicted labels: {len(np.unique(predicted_labels))}")

    # -----------------------------
    # Confidence analysis dataframe
    # -----------------------------
    top1_conf = np.max(probs, axis=1)
    sorted_probs = np.sort(probs, axis=1)
    top2_conf = sorted_probs[:, -2] if probs.shape[1] > 1 else np.zeros_like(top1_conf)
    margins = top1_conf - top2_conf
    correct = predicted_labels == labels

    confidence_df = pd.DataFrame({
        "true_label_id": labels,
        "pred_label_id": predicted_labels,
        "true_label": [id2label_mapping[int(i)] for i in labels],
        "pred_label": [id2label_mapping[int(i)] for i in predicted_labels],
        "top1_confidence": top1_conf,
        "top2_confidence": top2_conf,
        "top1_top2_margin": margins,
        "correct": correct,
    })

    if extra_prediction_columns is not None:
        for col_name, values in extra_prediction_columns.items():
            confidence_df[col_name] = values

    print("\nConfidence summary:")
    display(
        confidence_df.groupby("correct")[[
            "top1_confidence",
            "top1_top2_margin"
        ]].agg(["mean", "median", "min", "max", "count"]).round(4)
    )

    print("\nMost confident wrong predictions:")
    wrong_predictions = confidence_df[confidence_df["correct"] == False]
    if len(wrong_predictions) > 0:
        display(
            wrong_predictions
            .sort_values("top1_confidence", ascending=False)
            .head(10)
            .round(4)
        )
    else:
        print("No wrong predictions found.")

    print("\nLowest-margin predictions:")
    display(
        confidence_df
        .sort_values("top1_top2_margin", ascending=True)
        .head(10)
        .round(4)
    )

    # -----------------------------
    # Full confusion matrix
    # -----------------------------
    all_label_ids = list(range(num_labels))
    target_names = [id2label_mapping[i] for i in all_label_ids]

    cm = confusion_matrix(
        labels,
        predicted_labels,
        labels=all_label_ids,
    )

    if normalize_cm:
        cm_to_plot = cm.astype("float") / cm.sum(axis=1, keepdims=True)
        cm_to_plot = np.nan_to_num(cm_to_plot)
    else:
        cm_to_plot = cm

    plt.figure(figsize=(12, 10))

    if num_labels > 20:
        sns.heatmap(
            cm_to_plot,
            cmap="Blues",
            xticklabels=False,
            yticklabels=False,
            cbar=True,
        )
        plt.title(
            f"Confusion Matrix for {model_display_name} on {dataset_name}\n"
            f"Labels omitted because there are {num_labels} classes"
        )
    else:
        sns.heatmap(
            cm_to_plot,
            annot=True,
            fmt=".2f" if normalize_cm else "g",
            cmap="Blues",
            xticklabels=target_names,
            yticklabels=target_names,
        )
        plt.title(f"Confusion Matrix for {model_display_name} on {dataset_name}")

    plt.xlabel("Predicted label")
    plt.ylabel("True label")
    plt.tight_layout()
    plt.show()

    # -----------------------------
    # Classification report
    # -----------------------------
    print(f"\n--- Per-class metrics for {model_display_name} ---")

    report = classification_report(
        labels,
        predicted_labels,
        labels=all_label_ids,
        target_names=target_names,
        output_dict=True,
        zero_division=0,
    )

    report_df = pd.DataFrame(report).transpose()

    class_metrics_df = report_df.loc[target_names].copy()
    class_metrics_df["f1-score"] = pd.to_numeric(class_metrics_df["f1-score"], errors="coerce")

    print("\nTop 10 weakest classes by F1-score:")
    weakest_classes = class_metrics_df.sort_values("f1-score", ascending=True).head(10)
    display(weakest_classes[["precision", "recall", "f1-score", "support"]].round(4))

    print("\nTop 10 strongest classes by F1-score:")
    strongest_classes = class_metrics_df.sort_values("f1-score", ascending=False).head(10)
    display(strongest_classes[["precision", "recall", "f1-score", "support"]].round(4))

    metrics["classification_report_df"] = report_df
    metrics["confidence_df"] = confidence_df
    metrics["logits"] = logits
    metrics["labels"] = labels

    return metrics


In [9]:
from datasets import load_dataset, DatasetDict
from collections import defaultdict, Counter
import random
import numpy as np

# -----------------------------
# Balanced Food101 subset config
# -----------------------------
DATASET_NAME = "ethz/food101"
NUM_SELECTED_CLASSES = 20
SAMPLES_PER_CLASS = 250
TEST_SIZE = 0.20
SEED = 42

# Load the full Food101 training split.
# We use the train split and create our own stratified train/validation split
# because this experiment is meant to compare fine-tuning strategies cheaply.
food_full_train = load_dataset(DATASET_NAME, split="train")
original_labels = food_full_train.features["label"].names

rng = random.Random(SEED)

# Group dataset indices by original Food101 class id
label_to_indices = defaultdict(list)
for idx, label_id in enumerate(food_full_train["label"]):
    label_to_indices[int(label_id)].append(idx)

# Keep only classes that have enough examples
eligible_label_ids = [
    label_id
    for label_id, indices in label_to_indices.items()
    if len(indices) >= SAMPLES_PER_CLASS
]

if len(eligible_label_ids) < NUM_SELECTED_CLASSES:
    raise ValueError(
        f"Only {len(eligible_label_ids)} classes have at least "
        f"{SAMPLES_PER_CLASS} samples. Need {NUM_SELECTED_CLASSES}."
    )

# Select 20 classes reproducibly
selected_old_label_ids = sorted(rng.sample(eligible_label_ids, NUM_SELECTED_CLASSES))

# Select exactly 250 images per selected class
selected_indices = []
for old_label_id in selected_old_label_ids:
    indices = label_to_indices[old_label_id].copy()
    rng.shuffle(indices)
    selected_indices.extend(indices[:SAMPLES_PER_CLASS])

rng.shuffle(selected_indices)

food_subset = food_full_train.select(selected_indices)

print(f"Total selected images: {len(food_subset)}")
print(f"Selected classes: {len(selected_old_label_ids)}")
print("Selected class names:")
for old_label_id in selected_old_label_ids:
    print(f"  old_id={old_label_id:3d} -> {original_labels[old_label_id]}")

# Stratified split while labels are still original Food101 label IDs
food = food_subset.train_test_split(
    test_size=TEST_SIZE,
    shuffle=True,
    seed=SEED,
    stratify_by_column="label",
)

# Remap selected original labels to contiguous labels 0..19.
# This is important because the classifier head will have exactly 20 outputs.
old_to_new = {
    old_label_id: new_label_id
    for new_label_id, old_label_id in enumerate(selected_old_label_ids)
}

new_to_old = {
    new_label_id: old_label_id
    for old_label_id, new_label_id in old_to_new.items()
}

labels = [
    original_labels[new_to_old[new_label_id]]
    for new_label_id in range(NUM_SELECTED_CLASSES)
]

id2label = {
    new_label_id: label_name
    for new_label_id, label_name in enumerate(labels)
}

label2id = {
    label_name: new_label_id
    for new_label_id, label_name in id2label.items()
}

def remap_label(example):
    example["label"] = old_to_new[int(example["label"])]
    return example

food = DatasetDict({
    "train": food["train"].map(remap_label),
    "test": food["test"].map(remap_label),
})

# Sanity checks
train_counts = Counter(food["train"]["label"])
test_counts = Counter(food["test"]["label"])

print("\nAfter remapping:")
print(f"Train size: {len(food['train'])}")
print(f"Validation size: {len(food['test'])}")
print(f"Number of labels: {len(labels)}")
print(f"Train class counts: {sorted(train_counts.items())}")
print(f"Validation class counts: {sorted(test_counts.items())}")

assert len(food["train"]) + len(food["test"]) == NUM_SELECTED_CLASSES * SAMPLES_PER_CLASS
assert set(train_counts.keys()) == set(range(NUM_SELECTED_CLASSES))
assert set(test_counts.keys()) == set(range(NUM_SELECTED_CLASSES))


README.md:   0%|          | 0.00/16.4k [00:00<?, ?B/s]

data/train-00000-of-00008.parquet:   0%|          | 0.00/490M [00:00<?, ?B/s]

data/train-00001-of-00008.parquet:   0%|          | 0.00/464M [00:00<?, ?B/s]

data/train-00002-of-00008.parquet:   0%|          | 0.00/472M [00:00<?, ?B/s]

data/train-00003-of-00008.parquet:   0%|          | 0.00/464M [00:00<?, ?B/s]

...

Generating validation split:   0%|          | 0/25250 [00:00<?, ? examples/s]

Total selected images: 5000
Selected classes: 20
Selected class names:
  old_id= 10 -> bruschetta
  old_id= 11 -> caesar_salad
  old_id= 13 -> caprese_salad
  old_id= 15 -> ceviche
  old_id= 21 -> chocolate_cake
  old_id= 24 -> clam_chowder
  old_id= 31 -> donuts
  old_id= 33 -> edamame
  old_id= 34 -> eggs_benedict
  old_id= 40 -> french_fries
  old_id= 44 -> fried_rice
  old_id= 53 -> hamburger
  old_id= 54 -> hot_and_sour_soup
  old_id= 58 -> ice_cream
  old_id= 59 -> lasagna
  old_id= 64 -> miso_soup
  old_id= 72 -> pancakes
  old_id= 74 -> peking_duck
  old_id= 97 -> takoyaki
  old_id= 99 -> tuna_tartare


Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]


After remapping:
Train size: 4000
Validation size: 1000
Number of labels: 20
Train class counts: [(0, 200), (1, 200), (2, 200), (3, 200), (4, 200), (5, 200), (6, 200), (7, 200), (8, 200), (9, 200), (10, 200), (11, 200), (12, 200), (13, 200), (14, 200), (15, 200), (16, 200), (17, 200), (18, 200), (19, 200)]
Validation class counts: [(0, 50), (1, 50), (2, 50), (3, 50), (4, 50), (5, 50), (6, 50), (7, 50), (8, 50), (9, 50), (10, 50), (11, 50), (12, 50), (13, 50), (14, 50), (15, 50), (16, 50), (17, 50), (18, 50), (19, 50)]


In [10]:
# Label mappings for the 20-class balanced Food101 subset.
# These are already created in the dataset-loading cell above.
# The classifier head should use len(labels) == 20.

print(f"Number of selected labels: {len(labels)}")
print("id2label:")
for class_id, class_name in id2label.items():
    print(f"  {class_id}: {class_name}")

print("\nlabel2id:")
for class_name, class_id in label2id.items():
    print(f"  {class_name}: {class_id}")


Number of selected labels: 20
id2label:
  0: bruschetta
  1: caesar_salad
  2: caprese_salad
  3: ceviche
  4: chocolate_cake
  5: clam_chowder
  6: donuts
  7: edamame
  8: eggs_benedict
  9: french_fries
  10: fried_rice
  11: hamburger
  12: hot_and_sour_soup
  13: ice_cream
  14: lasagna
  15: miso_soup
  16: pancakes
  17: peking_duck
  18: takoyaki
  19: tuna_tartare

label2id:
  bruschetta: 0
  caesar_salad: 1
  caprese_salad: 2
  ceviche: 3
  chocolate_cake: 4
  clam_chowder: 5
  donuts: 6
  edamame: 7
  eggs_benedict: 8
  french_fries: 9
  fried_rice: 10
  hamburger: 11
  hot_and_sour_soup: 12
  ice_cream: 13
  lasagna: 14
  miso_soup: 15
  pancakes: 16
  peking_duck: 17
  takoyaki: 18
  tuna_tartare: 19


In [11]:
import os
import torch

def conditional_train_model(model, trainer, training_args, model_name):
    output_dir = training_args.output_dir
    last_checkpoint = None
    if os.path.isdir(output_dir) and not FORCE_FINE_TUNING:
        try:
            last_checkpoint = get_last_checkpoint(output_dir)
            print(f"Loading checkpoint for {model_name} from {last_checkpoint}")
            model = AutoModelForImageClassification.from_pretrained(last_checkpoint)
            # Ensure model is on the correct device if not already handled by from_pretrained
            if torch.cuda.is_available():
                model.to('cuda')
        except Exception as e:
            print(f"Could not load checkpoint for {model_name}: {e}. Retraining.")
            last_checkpoint = None

    if last_checkpoint is None or FORCE_FINE_TUNING:
        print(f"Fine-tuning {model_name}...")
        train_result = trainer.train(resume_from_checkpoint=last_checkpoint)
        trainer.save_model()
        metrics = train_result.metrics
        trainer.log_metrics("train", metrics)
        trainer.save_metrics("train", metrics)
        trainer.save_state()
        return metrics.get('train_runtime', 0.0)
    else:
        print(f"Skipping fine-tuning for {model_name}, loaded from checkpoint.")
        # If not fine-tuning, we might still want to run an evaluation to get metrics
        # or just return 0 for train_runtime as no training occurred.
        return 0.0

# Helper for get_last_checkpoint, usually from transformers.trainer_utils
def get_last_checkpoint(checkpoint_dir):
    checkpoints = [path for path in os.listdir(checkpoint_dir) if path.startswith("checkpoint-")]
    if not checkpoints:
        return None
    return os.path.join(checkpoint_dir, max(checkpoints, key=lambda x: int(x.split('-')[-1])))


In [12]:
from transformers import AutoImageProcessor, AutoModelForImageClassification

# ConvNeXt-Tiny model from Hugging Face
convnext_model_id_hf = "facebook/convnext-tiny-224"

# Load the ConvNeXt-specific image processor
convnext_image_processor = AutoImageProcessor.from_pretrained(convnext_model_id_hf)

# Apply ConvNeXt-specific transforms to the already split balanced Food101 subset
food_convnext = food.copy()

food_convnext["train"] = food_convnext["train"].with_transform(
    lambda examples: apply_transforms(
        examples,
        convnext_image_processor,
        is_train=True
    )
)

food_convnext["test"] = food_convnext["test"].with_transform(
    lambda examples: apply_transforms(
        examples,
        convnext_image_processor,
        is_train=False
    )
)

print("ConvNeXt-specific image processor and transforms defined and applied.")

preprocessor_config.json:   0%|          | 0.00/266 [00:00<?, ?B/s]

ConvNeXt-specific image processor and transforms defined and applied.


In [13]:

# Load the ConvNeXt-Tiny model with a new 20-class classification head
convnext_model_hf = AutoModelForImageClassification.from_pretrained(
    convnext_model_id_hf,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

# Ensure all parameters require gradients for full fine-tuning
for param in convnext_model_hf.parameters():
    param.requires_grad = True

print("ConvNeXt-Tiny model loaded and configured for full fine-tuning.")
print(convnext_model_hf)

# Sanity checks
print("Number of labels:", convnext_model_hf.config.num_labels)
print("id2label length:", len(convnext_model_hf.config.id2label))
print("label2id length:", len(convnext_model_hf.config.label2id))

config.json:   0%|          | 0.00/69.6k [00:00<?, ?B/s]

[transformers] You passed `num_labels=20` which is incompatible to the `id2label` map of length `1000`.


pytorch_model.bin:   0%|          | 0.00/114M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/182 [00:00<?, ?it/s]

[transformers] ConvNextForImageClassification LOAD REPORT from: facebook/convnext-tiny-224
Key               | Status   |                                                                                           
------------------+----------+-------------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([20])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([20, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


ConvNeXt-Tiny model loaded and configured for full fine-tuning.
ConvNextForImageClassification(
  (convnext): ConvNextModel(
    (embeddings): ConvNextEmbeddings(
      (patch_embeddings): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
      (layernorm): ConvNextLayerNorm((96,), eps=1e-06, elementwise_affine=True)
    )
    (encoder): ConvNextEncoder(
      (stages): ModuleList(
        (0): ConvNextStage(
          (downsampling_layer): ModuleList()
          (layers): ModuleList(
            (0-2): 3 x ConvNextLayer(
              (dwconv): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
              (layernorm): ConvNextLayerNorm((96,), eps=1e-06, elementwise_affine=True)
              (pwconv1): Linear(in_features=96, out_features=384, bias=True)
              (act): GELUActivation()
              (pwconv2): Linear(in_features=384, out_features=96, bias=True)
              (drop_path): Identity()
            )
          )
        )
        (1): ConvN

In [ ]:
from transformers import TrainingArguments, Trainer

# Define TrainingArguments for ConvNeXt-Tiny full fine-tuning
training_args_convnext_hf = TrainingArguments(
    output_dir="/content/drive/MyDrive/ViTExperiments/food101_20cls_250_convnext_tiny_full_finetune",
    remove_unused_columns=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-4,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    warmup_steps=10,
    logging_steps=10,
    run_name="food101_convnext_tiny_full_finetune",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=False,
)

# Initialize the Trainer for ConvNeXt-Tiny
trainer_convnext_hf = Trainer(
    model=convnext_model_hf,
    args=training_args_convnext_hf,
    data_collator=data_collator,
    train_dataset=food_convnext["train"],
    eval_dataset=food_convnext["test"],
    processing_class=convnext_image_processor,
    compute_metrics=compute_metrics,
)

print("Starting ConvNeXt-Tiny full model fine-tuning...")

convnext_hf_full_train_time = conditional_train_model(
    convnext_model_hf,
    trainer_convnext_hf,
    training_args_convnext_hf,
    "ConvNeXt-Tiny HF Full Fine-tune"
)

print("ConvNeXt-Tiny full model fine-tuning complete.")

model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

Starting ConvNeXt-Tiny full model fine-tuning...
Fine-tuning ConvNeXt-Tiny HF Full Fine-tune...


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1,F1,Top 5 Accuracy,Mean Top1 Confidence,Mean Top1 Top2 Margin,Correct Mean Confidence,Wrong Mean Confidence,Ece,Nll,Brier Score
1,2.220653,2.011777,0.701000,0.690031,0.690031,0.690031,0.931000,0.166429,0.079019,0.189487,0.112370,0.534571,2.011777,0.766373
2,1.330161,1.110025,0.806000,0.802105,0.802105,0.802105,0.960000,0.446136,0.340035,0.496346,0.237533,0.359864,1.110025,0.442604
3,0.804162,0.761839,0.840000,0.840928,0.840928,0.840928,0.971000,0.612630,0.511395,0.667192,0.326176,0.229829,0.761839,0.299120


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

### VLM Model Loading and Inference Setup

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

# SmolVLM model from Hugging Face.
# This is used as a zero-shot VLM classifier by scoring each candidate class label
# with image-conditioned log-likelihood, not by generating free-form text.
smolvlm_model_id = "HuggingFaceTB/SmolVLM-Instruct"

vlm_device = "cuda" if torch.cuda.is_available() else "cpu"
vlm_dtype = torch.float16 if vlm_device == "cuda" else torch.float32

processor_smolvlm = AutoProcessor.from_pretrained(smolvlm_model_id)
model_smolvlm = AutoModelForImageTextToText.from_pretrained(
    smolvlm_model_id,
    torch_dtype=vlm_dtype,
)
model_smolvlm.to(vlm_device)
model_smolvlm.eval()

print(f"SmolVLM processor and model loaded on {vlm_device} with dtype={vlm_dtype}.")

### VLM forced-choice class-logit scoring setup


In [ ]:
import torch.nn.functional as F
from transformers import AutoProcessor, AutoModelForImageTextToText
import matplotlib.pyplot as plt

def vlm_label_name(label_name):
    """Convert dataset label style to natural language label style for the VLM."""
    return str(label_name).replace("_", " ")


def build_vlm_classification_prompt(id2label_mapping):
    """
    Prompt used for forced-choice VLM classification.
    The candidate label probabilities are computed by scoring each possible
    answer string as a continuation of this prompt.
    """
    class_names = [vlm_label_name(id2label_mapping[i]) for i in range(len(id2label_mapping))]
    class_list = ", ".join(class_names)

    prompt = (
        "<image>\n"
        "You are a food image classifier. "
        "Choose the single best class for the image from this list: "
        f"{class_list}.\n"
        "Answer with exactly one class name.\n"
        "Answer:"
    )
    return prompt, class_names


def move_processor_inputs_to_device(inputs, device, dtype=None):
    """
    Moves processor outputs to device. Floating tensors such as pixel_values are
    cast to the model dtype when appropriate; integer token tensors stay integer.
    """
    moved = {}
    for key, value in inputs.items():
        value = value.to(device)
        if dtype is not None and torch.is_floating_point(value):
            value = value.to(dtype=dtype)
        moved[key] = value
    return moved


def score_vlm_candidates_for_image(
    model,
    processor,
    image,
    prompt,
    candidate_names,
    device,
    dtype=None,
    candidate_batch_size=20,
    normalize_by_answer_length=True,
):
    """
    Returns one scalar score per candidate class.

    For each candidate label, we compute the image-conditioned log-likelihood of
    the candidate text as the answer continuation. These scores act as class
    logits. Applying softmax over these scores gives a probability distribution
    over the same 20 classes used by ConvNeXt.

    This is a forced-choice VLM classification setup:
        image + prompt + candidate_label -> candidate score
    """
    image = image.convert("RGB")
    scores = []

    # Prompt length is needed so the loss/log-prob is computed only on the
    # candidate answer tokens, not on the prompt tokens.
    prompt_inputs = processor(
        images=image,
        text=prompt,
        return_tensors="pt",
        padding=True,
    )
    prompt_len = int(prompt_inputs["attention_mask"][0].sum().item())

    for start in range(0, len(candidate_names), candidate_batch_size):
        batch_candidates = candidate_names[start:start + candidate_batch_size]
        full_texts = [f"{prompt} {candidate}" for candidate in batch_candidates]
        images = [image] * len(full_texts)

        inputs = processor(
            images=images,
            text=full_texts,
            return_tensors="pt",
            padding=True,
        )
        inputs = move_processor_inputs_to_device(inputs, device=device, dtype=dtype)

        input_ids = inputs["input_ids"]
        attention_mask = inputs.get("attention_mask", torch.ones_like(input_ids))

        labels = input_ids.clone()
        labels[:, :prompt_len] = -100
        labels[attention_mask == 0] = -100

        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits.float()

        # Causal LM scoring: token t is predicted by logits at position t-1.
        shift_logits = logits[:, :-1, :]
        shift_labels = labels[:, 1:]
        answer_mask = shift_labels.ne(-100)

        safe_labels = shift_labels.masked_fill(~answer_mask, 0)
        token_log_probs = F.log_softmax(shift_logits, dim=-1)
        selected_token_log_probs = token_log_probs.gather(
            dim=-1,
            index=safe_labels.unsqueeze(-1),
        ).squeeze(-1)

        token_counts = answer_mask.sum(dim=1).clamp(min=1)
        sum_log_probs = (selected_token_log_probs * answer_mask).sum(dim=1)

        if normalize_by_answer_length:
            batch_scores = sum_log_probs / token_counts
        else:
            batch_scores = sum_log_probs

        scores.extend(batch_scores.detach().cpu().numpy().tolist())

    return np.asarray(scores, dtype=np.float32)


def get_vlm_logits_and_labels(
    model,
    processor,
    dataset,
    id2label_mapping,
    device=None,
    dtype=None,
    candidate_batch_size=20,
    max_eval_samples=None,
    normalize_by_answer_length=True,
):
    """
    Converts VLM forced-choice classification into the same output type as a
    standard classifier:
        logits: shape [num_samples, num_classes]
        labels: shape [num_samples]

    This allows the exact same evaluation pipeline to be used for ConvNeXt and
    SmolVLM.
    """
    if device is None:
        device = next(model.parameters()).device
    if dtype is None:
        dtype = next(model.parameters()).dtype

    prompt, candidate_names = build_vlm_classification_prompt(id2label_mapping)
    num_classes = len(candidate_names)

    total = len(dataset) if max_eval_samples is None else min(len(dataset), max_eval_samples)
    logits = np.zeros((total, num_classes), dtype=np.float32)
    labels = np.zeros(total, dtype=np.int64)

    model.eval()

    print("VLM forced-choice prompt:")
    print(prompt)
    print(f"\nScoring {total} images against {num_classes} candidate labels...")

    for i in range(total):
        item = dataset[i]
        image = item["image"]
        labels[i] = int(item["label"])

        logits[i] = score_vlm_candidates_for_image(
            model=model,
            processor=processor,
            image=image,
            prompt=prompt,
            candidate_names=candidate_names,
            device=device,
            dtype=dtype,
            candidate_batch_size=candidate_batch_size,
            normalize_by_answer_length=normalize_by_answer_length,
        )

        if (i + 1) % 25 == 0 or (i + 1) == total:
            print(f"Processed {i + 1}/{total} VLM samples")

    return logits, labels

print("VLM forced-choice class-logit scoring functions defined.")


## Model Evaluation and Comparison

In [ ]:
# -----------------------------
# Centralized model evaluation
# -----------------------------
# ConvNeXt and SmolVLM are both evaluated by producing logits of shape [N, 20]
# and passing those logits into the same evaluate_logits_and_plot(...) function.

# Evaluate the fine-tuned ConvNeXt-Tiny model.
print("\n--- Running ConvNeXt-Tiny Fine-tuned Evaluation ---")
convnext_logits, convnext_labels, convnext_eval_loss = get_trainer_logits_and_labels(
    trainer_convnext_hf,
    food_convnext["test"],
)

convnext_metrics = evaluate_logits_and_plot(
    logits=convnext_logits,
    labels=convnext_labels,
    id2label_mapping=id2label,
    dataset_name="Food101 20-Class Validation Set",
    num_labels=len(labels),
    model_display_name="ConvNeXt-Tiny HF Full Fine-tune",
    training_time=convnext_hf_full_train_time,
    eval_loss=convnext_eval_loss,
    normalize_cm=True,
)

# Evaluate the zero-shot SmolVLM model using forced-choice class logits.
print("\n--- Running SmolVLM Forced-Choice Evaluation ---")

# Set to a small integer such as 100 for a quick smoke test, or None for the full validation set.
VLM_MAX_EVAL_SAMPLES = None

vlm_logits, vlm_labels = get_vlm_logits_and_labels(
    model=model_smolvlm,
    processor=processor_smolvlm,
    dataset=food["test"],
    id2label_mapping=id2label,
    device=vlm_device,
    dtype=vlm_dtype,
    candidate_batch_size=2,
    max_eval_samples=VLM_MAX_EVAL_SAMPLES,
    normalize_by_answer_length=True,
)

vlm_metrics = evaluate_logits_and_plot(
    logits=vlm_logits,
    labels=vlm_labels,
    id2label_mapping=id2label,
    dataset_name="Food101 20-Class Validation Set",
    num_labels=len(labels),
    model_display_name="SmolVLM-Instruct Zero-Shot Forced-Choice",
    training_time=None,
    eval_loss=None,
    normalize_cm=True,
)

# Store numeric results for comparison.
all_results = {
    "ConvNeXt-Tiny HF Full Fine-tune": {
        k: v for k, v in convnext_metrics.items() if isinstance(v, (int, float, np.integer, np.floating))
    },
    "SmolVLM-Instruct Zero-Shot Forced-Choice": {
        k: v for k, v in vlm_metrics.items() if isinstance(v, (int, float, np.integer, np.floating))
    },
}

comparison_df = pd.DataFrame(all_results).T

print("\n--- Final Model Comparison ---")
display(comparison_df.round(4))
